Imports (RUN FIRST!)

In [ ]:
import http.client
import urllib.request
import urllib.parse
import hashlib
import hmac
import base64
import json
import time
from dotenv import load_dotenv
import os
import requests
from IPython.display import display

load_dotenv()




Current Bitcoin Balance

In [ ]:
# Get all asset balances.

def main():
   response = request(
      method="POST",
      path="/0/private/Balance",
      public_key=os.getenv("API_KEY"),
      private_key=os.getenv("PRIVATE_KEY"),
      environment="https://api.kraken.com",
   )
   print(response.read().decode())

def request(method: str = "GET", path: str = "", query: dict | None = None, body: dict | None = None, public_key: str = "", private_key: str = "", environment: str = "") -> http.client.HTTPResponse:
   url = environment + path
   query_str = ""
   if query is not None and len(query) > 0:
      query_str = urllib.parse.urlencode(query)
      url += "?" + query_str
   nonce = ""
   if len(public_key) > 0:
      if body is None:
         body = {}
      nonce = body.get("nonce")
      if nonce is None:
         nonce = get_nonce()
         body["nonce"] = nonce
   headers = {}
   body_str = ""
   if body is not None and len(body) > 0:
      body_str = json.dumps(body)
      headers["Content-Type"] = "application/json"
   if len(public_key) > 0:
      headers["API-Key"] = public_key
      headers["API-Sign"] = get_signature(private_key, query_str+body_str, nonce, path)
   req = urllib.request.Request(
      method=method,
      url=url,
      data=body_str.encode(),
      headers=headers,
   )
   return urllib.request.urlopen(req)

def get_nonce() -> str:
   return str(int(time.time() * 1000))

def get_signature(private_key: str, data: str, nonce: str, path: str) -> str:
   return sign(
      private_key=private_key,
      message=path.encode() + hashlib.sha256(
            (nonce + data)
         .encode()
      ).digest()
   )

def sign(private_key: str, message: bytes) -> str:
   return base64.b64encode(
      hmac.new(
         key=base64.b64decode(private_key),
         msg=message,
         digestmod=hashlib.sha512,
      ).digest()
   ).decode()


if __name__ == "__main__":
   main()

Trade History

In [ ]:
# Top 50 most recent account trades.
def main():
   response = request(
      method="POST",
      path="/0/private/TradesHistory",
      public_key=os.getenv("API_KEY"),
      private_key=os.getenv("PRIVATE_KEY"),
      environment="https://api.kraken.com",
   )
   raw = response.read()
   data = json.loads(raw)

   trades = data["result"]["trades"]

   import pandas as pd

   df = pd.DataFrame.from_dict(trades, orient="index")
   df["time"] = pd.to_datetime(df["time"], unit="s")
   df = df.sort_values("time", ascending=False)
   df = df[[
    "pair",
    "time",
    "type",
    "ordertype",
    "tradeordertype",
    "price",
    "cost",
    "fee",
    "vol"
]]
   display(df)

def request(method: str = "GET", path: str = "", query: dict | None = None, body: dict | None = None, public_key: str = "", private_key: str = "", environment: str = "") -> http.client.HTTPResponse:
   url = environment + path
   query_str = ""
   if query is not None and len(query) > 0:
      query_str = urllib.parse.urlencode(query)
      url += "?" + query_str
   nonce = ""
   if len(public_key) > 0:
      if body is None:
         body = {}
      nonce = body.get("nonce")
      if nonce is None:
         nonce = get_nonce()
         body["nonce"] = nonce
   headers = {}
   body_str = ""
   if body is not None and len(body) > 0:
      body_str = json.dumps(body)
      headers["Content-Type"] = "application/json"
   if len(public_key) > 0:
      headers["API-Key"] = public_key
      headers["API-Sign"] = get_signature(private_key, query_str+body_str, nonce, path)
   req = urllib.request.Request(
      method=method,
      url=url,
      data=body_str.encode(),
      headers=headers,
   )
   return urllib.request.urlopen(req)

def get_nonce() -> str:
   return str(int(time.time() * 1000))

def get_signature(private_key: str, data: str, nonce: str, path: str) -> str:
   return sign(
      private_key=private_key,
      message=path.encode() + hashlib.sha256(
            (nonce + data)
         .encode()
      ).digest()
   )

def sign(private_key: str, message: bytes) -> str:
   return base64.b64encode(
      hmac.new(
         key=base64.b64decode(private_key),
         msg=message,
         digestmod=hashlib.sha512,
      ).digest()
   ).decode()


if __name__ == "__main__":
   main()


In [ ]:
# Ledger

import http.client
import urllib.request
import urllib.parse
import hashlib
import hmac
import base64
import json
import time
from dotenv import load_dotenv
import os
from IPython.display import display
import pandas as pd

load_dotenv()

def main():

    response = request(
        method="POST",
        path="/0/private/Ledgers",
        public_key=os.getenv("API_KEY"),
        private_key=os.getenv("PRIVATE_KEY"),
        environment="https://api.kraken.com",
    )

    raw = response.read()
    data = json.loads(raw)

    ledger = data["result"]["ledger"]

    df = pd.DataFrame.from_dict(ledger, orient="index")

    # Convert timestamp
    df["time"] = pd.to_datetime(df["time"], unit="s")

    # Sort newest first
    df = df.sort_values("time", ascending=False)

    # Select useful columns
    df = df[[
        "time",
        "type",
        "asset",
        "amount",
        "fee",
        "balance"
    ]]

    display(df.head(2))


def request(
    method: str = "GET",
    path: str = "",
    query: dict | None = None,
    body: dict | None = None,
    public_key: str = "",
    private_key: str = "",
    environment: str = ""
) -> http.client.HTTPResponse:

    url = environment + path

    query_str = ""

    if query is not None and len(query) > 0:
        query_str = urllib.parse.urlencode(query)
        url += "?" + query_str

    nonce = ""

    if len(public_key) > 0:

        if body is None:
            body = {}

        nonce = body.get("nonce")

        if nonce is None:
            nonce = get_nonce()
            body["nonce"] = nonce

    headers = {}

    body_str = ""

    if body is not None and len(body) > 0:
        body_str = json.dumps(body)
        headers["Content-Type"] = "application/json"

    if len(public_key) > 0:
        headers["API-Key"] = public_key
        headers["API-Sign"] = get_signature(
            private_key,
            query_str + body_str,
            nonce,
            path
        )

    req = urllib.request.Request(
        method=method,
        url=url,
        data=body_str.encode(),
        headers=headers,
    )

    return urllib.request.urlopen(req)


def get_nonce() -> str:
    return str(int(time.time() * 1000))


def get_signature(
    private_key: str,
    data: str,
    nonce: str,
    path: str
) -> str:

    return sign(
        private_key=private_key,
        message=path.encode() + hashlib.sha256(
            (nonce + data).encode()
        ).digest()
    )


def sign(private_key: str, message: bytes) -> str:

    return base64.b64encode(
        hmac.new(
            key=base64.b64decode(private_key),
            msg=message,
            digestmod=hashlib.sha512,
        ).digest()
    ).decode()


if __name__ == "__main__":
    main()

Current BTC Price

In [ ]:
import requests

url = "https://api.kraken.com/0/public/Ticker?pair=XBTCAD"

response = requests.get(url)
data = response.json()

print(data)

current_price = float(data["result"]["XXBTZCAD"]["c"][0])
print(current_price)

Buy Order (market ordertype)

In [ ]:
import http.client
import urllib.request
import urllib.parse
import hashlib
import hmac
import base64
import json
import time
import requests
import os
import sys


def main():

   trigger_price = float(os.getenv("TRIGGER_PRICE"))
   amount_invested = 1.00 
   MIN_BTC = 0.0001

   while True:

      url = "https://api.kraken.com/0/public/Ticker?pair=XXBTZCAD"

      response = requests.get(url)
      data = response.json()

      current_price = float(data["result"]["XXBTZCAD"]["c"][0])
      print("Current price:", current_price)

      if current_price <= trigger_price:
         # "volume": str(btc_amount),
         min_cad = MIN_BTC * current_price
         if amount_invested < min_cad:
            print(f"amount_invested too low. Need at least ${min_cad:.2f} CAD. Adjusting...")
            amount_invested = min_cad
         response = request(
            method="POST",
            path="/0/private/AddOrder",
            body={
               "ordertype": "market",
               "type": "buy",
               "volume": str(amount_invested),
               "pair": "XXBTZCAD",
               "oflags": "viqc"
            },
            public_key=os.getenv("API_KEY"),
            private_key=os.getenv("PRIVATE_KEY"),
            environment="https://api.kraken.com",
         )

         print(response.read().decode())
         print("ORDER PLACED. EXITING LOOP.")
         sys.exit(0) # exit safely with Railway Deployment to avoid a duplicate order

      else:
         print("Condition not met. Waiting...")

      time.sleep(10)  # avoid spamming API


def request(method: str = "GET", path: str = "", query: dict | None = None,
            body: dict | None = None, public_key: str = "",
            private_key: str = "", environment: str = "") -> http.client.HTTPResponse:

   url = environment + path
   query_str = ""

   if query is not None and len(query) > 0:
      query_str = urllib.parse.urlencode(query)
      url += "?" + query_str

   nonce = ""

   if len(public_key) > 0:
      if body is None:
         body = {}
      nonce = body.get("nonce")
      if nonce is None:
         nonce = get_nonce()
         body["nonce"] = nonce

   headers = {}
   body_str = ""

   if body is not None and len(body) > 0:
      body_str = json.dumps(body)
      headers["Content-Type"] = "application/json"

   if len(public_key) > 0:
      headers["API-Key"] = public_key
      headers["API-Sign"] = get_signature(private_key, query_str + body_str, nonce, path)

   req = urllib.request.Request(
      method=method,
      url=url,
      data=body_str.encode(),
      headers=headers,
   )

   return urllib.request.urlopen(req)


def get_nonce() -> str:
   return str(int(time.time() * 1000))


def get_signature(private_key: str, data: str, nonce: str, path: str) -> str:
   return sign(
      private_key=private_key,
      message=path.encode() + hashlib.sha256(
         (nonce + data).encode()
      ).digest()
   )


def sign(private_key: str, message: bytes) -> str:
   return base64.b64encode(
      hmac.new(
         key=base64.b64decode(private_key),
         msg=message,
         digestmod=hashlib.sha512,
      ).digest()
   ).decode()


if __name__ == "__main__":
   main()

Buy Order (limited ordertype)

In [ ]:
import http.client
import urllib.request
import urllib.parse
import hashlib
import hmac
import base64
import json
import time
import requests
import os


def main():

   trigger_price = float(os.getenv("TRIGGER_PRICE"))
   amount_invested = 1.00  
   MIN_BTC = 0.0001
   check_count = 0
   
   while True:

      url = "https://api.kraken.com/0/public/Ticker?pair=XXBTZCAD"

      response = requests.get(url)
      data = response.json()

      current_price = float(data["result"]["XXBTZCAD"]["c"][0])

      check_count += 1
      if check_count % 30 == 0:  # print every 30 checks × 10 seconds = 5 minutes
         print("Current price:", current_price)

      if current_price <= trigger_price:
         btc_amount = round(amount_invested / current_price, 8)

         if btc_amount < MIN_BTC:
            print(f"amount_invested too low. Need at least ${MIN_BTC * current_price:.2f} CAD. Adjusting...")
            btc_amount = MIN_BTC

         response = request(
            method="POST",
            path="/0/private/AddOrder",
            body={
                  "ordertype": "limit",
                  "type": "buy",
                  "volume": str(btc_amount),
                  "pair": "XXBTZCAD",
                  "price": str(trigger_price),
            },
            public_key=os.getenv("API_KEY"),
            private_key=os.getenv("PRIVATE_KEY"),
            environment="https://api.kraken.com",
         )

         print(response.read().decode())
         print("ORDER PLACED. EXITING LOOP.")
         sys.exit(0) # exit safely with Railway Deployment to avoid a duplicate order

      # else:
      #    print("Condition not met. Waiting...")

      time.sleep(10)  # avoid spamming API


def request(method: str = "GET", path: str = "", query: dict | None = None,
            body: dict | None = None, public_key: str = "",
            private_key: str = "", environment: str = "") -> http.client.HTTPResponse:

   url = environment + path
   query_str = ""

   if query is not None and len(query) > 0:
      query_str = urllib.parse.urlencode(query)
      url += "?" + query_str

   nonce = ""

   if len(public_key) > 0:
      if body is None:
         body = {}
      nonce = body.get("nonce")
      if nonce is None:
         nonce = get_nonce()
         body["nonce"] = nonce

   headers = {}
   body_str = ""

   if body is not None and len(body) > 0:
      body_str = json.dumps(body)
      headers["Content-Type"] = "application/json"

   if len(public_key) > 0:
      headers["API-Key"] = public_key
      headers["API-Sign"] = get_signature(private_key, query_str + body_str, nonce, path)

   req = urllib.request.Request(
      method=method,
      url=url,
      data=body_str.encode(),
      headers=headers,
   )

   return urllib.request.urlopen(req)


def get_nonce() -> str:
   return str(int(time.time() * 1000))


def get_signature(private_key: str, data: str, nonce: str, path: str) -> str:
   return sign(
      private_key=private_key,
      message=path.encode() + hashlib.sha256(
         (nonce + data).encode()
      ).digest()
   )


def sign(private_key: str, message: bytes) -> str:
   return base64.b64encode(
      hmac.new(
         key=base64.b64decode(private_key),
         msg=message,
         digestmod=hashlib.sha512,
      ).digest()
   ).decode()


if __name__ == "__main__":
   main()